%md
## Read: text PDFs (pdfplumber)
Extracts every table from PDFs routed to `text` in pdf_profile into `pdf_rows`
(same shape as the OCR reader). Rows are stored exactly as read; cleanup happens in silver.

**2026 result:** 225 of 225 files read, 86,693 rows; 218 files have one consistent width.
- AC178–183 bundle round-wise counting sheets and a candidate-by-round summary before
  the Form 20 booth sheet → silver must classify tables
- AC035 page 2 (rotated 90°) was read transposed → silver flips it back
- Layouts differ by district: NOTA position, whether "total valid" includes NOTA,
  extra columns (PS name, electors, turnout %) → silver infers layout from the tallies

In [0]:
%pip install pdfplumber

In [0]:
import io, pdfplumber

def read_text_tables(content):
    out = []
    with pdfplumber.open(io.BytesIO(content)) as pdf:
        for page_no, page in enumerate(pdf.pages):
            for t_idx, table in enumerate(page.extract_tables()):
                for r_idx, row in enumerate(table):
                    cells = [(c or "").replace("\n", " ").strip() for c in row]
                    out.append((page_no, t_idx, r_idx, cells))
    return out

In [0]:
from collections import Counter

pdf_files = spark.table("tnelections.form20_bronze.pdf_files")

for ac in [16, 54, 101, 148]:
    content = pdf_files.filter(f"year = 2026 AND ac_no = {ac}").select("content").first().content
    rows = read_text_tables(content)
    widths = Counter(len(r[3]) for r in rows)
    booth_like = [r[3] for r in rows if r[3] and r[3][0].isdigit()]
    print(f"AC{ac:03d}: {len(rows)} rows, widths {dict(widths)}")
    for cells in booth_like[:2]:
        print("   ", cells)
    print()

In [0]:
from pyspark.sql.functions import udf, col, explode, size, lit, current_timestamp
from pyspark.sql.types import ArrayType, StructType, StructField, IntegerType, StringType

row_schema = ArrayType(StructType([
    StructField("page", IntegerType()),
    StructField("table_idx", IntegerType()),
    StructField("row_idx", IntegerType()),
    StructField("cells", ArrayType(StringType())),
]))

def read_text_tables_safe(content):
    try:
        return read_text_tables(content)
    except Exception:
        return []          # a failed file shows up as missing in the check below

read_udf = udf(read_text_tables_safe, row_schema).asNondeterministic()

text_targets = (
    spark.table("tnelections.form20_bronze.pdf_profile")
    .filter("read_method = 'text'")
    .select("year", "ac_no")
)

text_rows = (
    spark.table("tnelections.form20_bronze.pdf_files")
    .join(text_targets, ["year", "ac_no"])
    .repartition(32)
    .withColumn("parsed", read_udf(col("content")))
    .withColumn("r", explode("parsed"))
    .select("year", "ac_no",
            col("r.page").alias("page"),
            col("r.table_idx").alias("table_idx"),
            col("r.row_idx").alias("row_idx"),
            col("r.cells").alias("cells"))
    .withColumn("n_cells", size("cells"))
    .withColumn("read_method", lit("text"))
    .withColumn("read_at", current_timestamp())
)

(text_rows.write
    .mode("overwrite")
    .option("replaceWhere", "read_method = 'text'")
    .saveAsTable("tnelections.form20_bronze.pdf_rows"))

In [0]:
display(spark.sql("""
    WITH per_file AS (
        SELECT ac_no, COUNT(*) AS rows, COUNT(DISTINCT n_cells) AS distinct_widths
        FROM tnelections.form20_bronze.pdf_rows
        WHERE read_method = 'text'
        GROUP BY ac_no
    )
    SELECT
        (SELECT COUNT(*) FROM tnelections.form20_bronze.pdf_profile
         WHERE read_method = 'text')          AS files_expected,
        COUNT(*)                              AS files_read,
        COUNT_IF(distinct_widths = 1)         AS one_width,
        COUNT_IF(distinct_widths > 1)         AS mixed_widths,
        SUM(rows)                             AS total_rows
    FROM per_file
"""))

In [0]:
display(spark.sql("""
    WITH mixed AS (
        SELECT ac_no
        FROM tnelections.form20_bronze.pdf_rows
        WHERE read_method = 'text'
        GROUP BY ac_no
        HAVING COUNT(DISTINCT n_cells) > 1
    )
    SELECT r.ac_no, r.n_cells,
           COUNT(*)                     AS rows,
           MIN(r.page)                  AS first_page,
           MAX(r.page)                  AS last_page,
           COUNT(DISTINCT r.page, r.table_idx) AS tables,
           any_value(r.cells)           AS sample_row
    FROM tnelections.form20_bronze.pdf_rows r
    JOIN mixed USING (ac_no)
    WHERE r.read_method = 'text'
    GROUP BY r.ac_no, r.n_cells
    ORDER BY r.ac_no, rows DESC
"""))